# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Matías Taborda
- Alumno 2 :    -

In [ ]:
import torch
# import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.datasets import fetch_lfw_people
# from sklearn.manifold import TSNE
# from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image
# import os
# import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__} | Dispositivo: {device}")

PyTorch: 2.11.0+cpu | Dispositivo: cpu


In [47]:
from PIL import Image
import torchvision.transforms as T

# Cargo el dataset LFW como base
lfw = fetch_lfw_people(min_faces_per_person=70, color=True, resize=1.0)

X_lfw = lfw.images    # (n_samples, 125, 94, 3) float en [0, 1]
y_lfw = lfw.target     # etiquetas numéricas
NOMBRES_LFW = lfw.target_names

print(f"Total de imágenes: {X_lfw.shape[0]}")
print(f"Tamaño de imagen: {X_lfw.shape[1:3]}")
print(f"Rango de valores: [{X_lfw.min():.2f}, {X_lfw.max():.2f}]")
print(f"\nPersonas ({len(NOMBRES_LFW)}):")

from collections import Counter
conteo = Counter(y_lfw)
for idx, nombre in enumerate(NOMBRES_LFW):
    print(f"  {nombre}: {conteo[idx]} imágenes")


# Convierto imágenes a PIL

X_lfw_pil = [Image.fromarray((img * 255).astype(np.uint8)) for img in X_lfw]

Total de imágenes: 1288
Tamaño de imagen: (125, 94)
Rango de valores: [0.00, 1.00]

Personas (7):
  Ariel Sharon: 77 imágenes
  Colin Powell: 236 imágenes
  Donald Rumsfeld: 121 imágenes
  George W Bush: 530 imágenes
  Gerhard Schroeder: 109 imágenes
  Hugo Chavez: 71 imágenes
  Tony Blair: 144 imágenes


In [48]:
# Verifico mi dataset 
from pathlib import Path
dataset_path = Path("src/data/dataset_tp1")
dataset_folders = []
 
for folder in dataset_path.iterdir():
    if folder.is_dir():
        dataset_folders.append(folder)

# verifico imágenes en las carpetas
print("*"*20)
print("Dataset TP1")
print("*"*20)
for folder in dataset_folders:
    print(f"{folder.name}: {len(list(folder.iterdir()))} imágenes")

********************
Dataset TP1
********************
gatacia: 2 imágenes
persona_1: 3 imágenes
persona_2: 4 imágenes


In [49]:
# Recorro las carpetas y cargo las imágenes 
X = []         
y = []
names = []

for persona_id, folder in enumerate(dataset_path.iterdir()):
    if folder.is_dir():
        names.append(folder.name)
        for img_file in folder.glob('*.jpg'):
            img = Image.open(img_file).convert('RGB')
            img = img.resize((94, 125)) # mismas dimensiones en LFW
            X.append(img) 
            y.append(persona_id) 
 

print(f"Dataset: {len(x)} imágenes, {len(names)} personas")

Dataset: 9 imágenes, 3 personas


In [67]:
if len(X) > 0:
    # datasets combinados
    X_combined = X_lfw_pil + X
    y_combined = list(y_lfw) + y
    combined_names = list(NOMBRES_LFW) + names
    print(f"Dataset obtenido: {len(X_combined)} imágenes | {len(combined_names)} personas")
else:
    print("Dataset vacío")

transform = T.Compose([
    T.ToTensor(),  # PIL -> Tensor
])

class FaceDataset(torch.utils.data.Dataset):
    def __init__(self, images_pil, labels, transform=None):
        self.images = images_pil
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

dataset_combined = FaceDataset(X_combined, y_combined, transform=transform)


Dataset obtenido: 1297 imágenes | 10 personas


In [70]:

# Divide con subset
train_size = int(0.8 * len(dataset_combined))
train_dataset = torch.utils.data.Subset(dataset_combined, range(train_size))
val_dataset = torch.utils.data.Subset(dataset_combined, range(train_size, len(dataset_combined)))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
 
print(f"Train: {len(train_dataset)} imágenes")
print(f"Val: {len(val_dataset)} imágenes")
print(f"Clases: {len(combined_names)}")

Train: 1037 imágenes
Val: 260 imágenes
Clases: 10
